In [113]:
#References: https://medium.com/@ayush.rane/experimenting-with-ml-trying-out-different-algorithms-for-one-simple-task-2431369c0223

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn import linear_model
from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.svm import SVC

In [50]:
# Load dataset
df = pd.read_csv("../data/df_pivot.csv")
print(f'Shape: {df.shape}')
df.head()


Shape: (1818, 54)


,year,stateabbr_c,statedesc_c,locationname_c,locationid_c,totalpopulation_c,geolocation_c,sca_release_id_c,msa_c,msa_code,...,pct_hispanic_u,pct_asian_u,CASTHMA,CHD,COPD,LPA,MHLTH,OBESITY,PHLTH,STROKE
0,2014,AK,Alaska,Anchorage,203000,"291,826","(61.149868731, -149.111113424)",2014_500_Cities_2016_release,"Anchorage, AK Metro Area",112.0,...,7.3,6.6,8.6,5.0,5.0,19.2,9.4,27.7,9.9,2.6
1,2014,AL,Alabama,Birmingham,107000,"212,237","(33.5275663773, -86.7988174678)",2014_500_Cities_2016_release,"Birmingham, AL Metro Area",136.0,...,NaN,NaN,11.4,7.6,9.4,31.7,17.0,39.0,18.3,5.0
2,2014,AL,Alabama,Huntsville,137000,"180,105","(34.6989692671, -86.6387042882)",2014_500_Cities_2016_release,"Huntsville, AL Metro Area",259.0,...,4.9,2.2,9.6,6.7,7.5,24.9,14.0,32.0,13.9,3.2
3,2014,AL,Alabama,Mobile,150000,"195,111","(30.6776248648, -88.1184482714)",2014_500_Cities_2016_release,"Mobile, AL Metro Area",333.0,...,2.6,1.9,10.7,7.6,8.7,27.4,15.8,37.6,16.3,4.1
4,2014,AL,Alabama,Montgomery,151000,"205,764","(32.3472645333, -86.2677059552)",2014_500_Cities_2016_release,"Montgomery, AL Metro Area",337.0,...,3.1,1.7,10.8,7.2,8.6,27.9,15.5,36.8,16.2,4.1


In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1818 entries, 0 to 1817
Data columns (total 54 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   year                       1818 non-null   int64  
 1   stateabbr_c                1818 non-null   str    
 2   statedesc_c                1818 non-null   str    
 3   locationname_c             1818 non-null   str    
 4   locationid_c               1818 non-null   int64  
 5   totalpopulation_c          1818 non-null   str    
 6   geolocation_c              1818 non-null   str    
 7   sca_release_id_c           1818 non-null   str    
 8   msa_c                      1818 non-null   str    
 9   msa_code                   1818 non-null   float64
 10  state_c                    1818 non-null   str    
 11  short_name_c               1818 non-null   str    
 12  source_dataset_c           1818 non-null   str    
 13  year_coverage_c            1818 non-null   int64  
 14  lat

In [52]:
missing = df.isnull().sum() / len(df) * 100
missing = missing[missing > 0].sort_values(ascending=False)
print('Features with missing values (% missing):')
print(missing.round(1).to_string())

Features with missing values (% missing):
tsun_n                       99.8
msa_u                        16.2
pct_hispanic_u               16.2
pct_black_u                  16.2
pct_white_u                  16.2
pct_less_than_hs_u           16.2
pct_graduate_degree_u        16.2
pct_bachelors_plus_u         16.2
median_household_income_u    16.2
cbsa_code_u                  16.2
pct_female_u                 16.2
pct_male_u                   16.2
median_age_u                 16.2
total_population_u           16.2
pct_asian_u                  16.2
cldd_n                        7.5
htdd_n                        7.4
tavg_n                        7.4
tmin_n                        7.0
dt00_n                        7.0
dt32_n                        7.0
emnt_n                        7.0
emxt_n                        6.9
dx90_n                        6.9
dx70_n                        6.9
dx32_n                        6.9
tmax_n                        6.9
prcp_n                        4.2
longit

In [53]:
df = df.drop(columns=["tsun_n"])

# Preparing the data with one hat encoding categorical 

In [57]:
# Separate feature types — keep as lists for ColumnTransformer
categorical_features = df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
numerical_features = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
                      if c != target]

# Mental Distress

MHLTH

In [90]:
target = "MHLTH"
X = df.drop(target, axis = 1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')


Train: 1454 samples
Test:  364 samples


I went through the Linear Models using sklearn: https://sklearn.org/stable/modules/linear_model.html. This is just an initial pass. There was a reason to skip classification models as said later due to having to make a second column like are we going to say "yes, you have mental health issues" or "no you dont". Then we could create a new column that we change based on values. But if that is not the case and we want to predict how bad the mental health is if you live in that state, might be more useful than a direct binary. 

In [135]:
# Creation of a dictionary to print at the end which we will sort by value 

# We use the R2 score to show the accuracy. We cannot use accuracy_score because we are
# are not using classification algorithms. The closer to 1 the R2 score is the better the
# algorithm is: https://www.geeksforgeeks.org/maths/r-squared/

R2_scores = {}
MSE_scores = {}

In [71]:
def make_preprocessor():
    """Returns a fresh ColumnTransformer for the Ames dataset."""
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
             categorical_features),
            ('num', SimpleImputer(strategy='mean'),
             numerical_features)
        ]
    )
    # Note: no remainder='passthrough' — all columns are explicitly listed

In [136]:
def create_ridge_pipeline(alpha):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        Ridge(alpha=alpha)
    )

# Checking it works
pipe = create_ridge_pipeline(1.0)
pipe.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe.predict(X_train))
print(f'Ridge training MSE (alpha=1): {mean_squared_error(y_train, pipe.predict(X_train)):.4f}')

MSE_scores["Ridge"] = MSE

y_pred = pipe.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Ridge: {accuracy}")
R2_scores["Ridge"] = accuracy

Ridge training MSE (alpha=1): 0.0735
R^2 score Ridge: 0.936354585225137


In [137]:
MSE_scores

{'Ridge': 0.07351334908908323}

In [ ]:
def create_krr_pipeline(alpha, gamma):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        KernelRidge(alpha=alpha, kernel='rbf', gamma=gamma)
    )

# Check it worked
pipe_krr = create_krr_pipeline(1.0, 0.1)
pipe_krr.fit(X_train, y_train)
print(f'KRR training MSE: {mean_squared_error(y_train, pipe_krr.predict(X_train)):.4f}')

y_pred = pipe_krr.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score KRR: {accuracy}")


KRR training MSE: 57.4452
R^2 score KRR: -29.654430137376345


In [116]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
alphas_krr = np.logspace(-5, 3, num=10)
gammas_krr = np.logspace(-8, -1, num=10)

mean_mses_2d = np.zeros((len(alphas_krr), len(gammas_krr)))

for i, alpha in enumerate(alphas_krr):
    for j, gamma in enumerate(gammas_krr):
        model = create_krr_pipeline(alpha, gamma)
        scores = cross_val_score(
            model, X_train, y_train,
            cv=kf, scoring=make_scorer(mean_squared_error)
        )
        mean_mse = scores.mean()
        mean_mses_2d[i, j] = mean_mse

best_i, best_j = np.unravel_index(np.argmin(mean_mses_2d), mean_mses_2d.shape)
best_alpha_krr = alphas_krr[best_i]
best_gamma_krr = gammas_krr[best_j]

print(f'Best alpha: {best_alpha_krr:.6f}')
print(f'Best gamma: {best_gamma_krr:.6f}')
print(f'CV MSE:     {mean_mses_2d[best_i, best_j]:.4f}')

Best alpha: 0.000010
Best gamma: 0.000002
CV MSE:     0.5673


In [140]:
# Using best gamma and alpha
pipe_krr = create_krr_pipeline(alpha=0.00001, gamma=0.000002)
pipe_krr.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe_krr.predict(X_train))
print(f'KRR training MSE: {mean_squared_error(y_train, pipe_krr.predict(X_train)):.4f}')
MSE_scores["KRR"] = MSE

y_pred = pipe_krr.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score KRR: {accuracy}")
R2_scores["KRR"] = accuracy

KRR training MSE: 0.0668
R^2 score KRR: 0.936316630905189


In [141]:
def create_linear_regression_pipeline():
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        LinearRegression()
    )

# Check it worked
pipe_linear_r = create_linear_regression_pipeline()
pipe_linear_r.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe_linear_r.predict(X_train))
print(f'Linear Regression training MSE: {mean_squared_error(y_train, pipe_linear_r.predict(X_train)):.4f}')
MSE_scores["Linear Regression"] = MSE

y_pred = pipe_linear_r.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Linear Regression: {accuracy}")
R2_scores["Linear Regression"] = accuracy

Linear Regression training MSE: 0.0726
R^2 score Linear Regression: 0.8887282513832646


In [142]:
def create_lasso_pipeline(alpha):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        linear_model.Lasso(alpha)
    )

# Check it worked
alpha=0.1
pipe_lasso = create_lasso_pipeline(alpha)
pipe_lasso.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe_lasso.predict(X_train))
print(f'Lasso training MSE: {mean_squared_error(y_train, pipe_lasso.predict(X_train)):.4f}')
MSE_scores["Lasso with alpha 0.1"] = MSE

y_pred = pipe_lasso.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Lasso: {accuracy} with alpha: {alpha}")
R2_scores["Lasso with alpha 0.1"] = accuracy

alpha_2=0.01
pipe_lasso = create_lasso_pipeline(alpha_2)
pipe_lasso.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe_lasso.predict(X_train))
print(f'Lasso training MSE: {mean_squared_error(y_train, pipe_lasso.predict(X_train)):.4f}')
MSE_scores["Lasso with alpha 0.01"] = MSE

y_pred = pipe_lasso.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Lasso: {accuracy} with alpha: {alpha_2}")
R2_scores["Lasso with alpha 0.01"] = accuracy

Lasso training MSE: 0.4521
R^2 score Lasso: 0.938032388831701 with alpha: 0.1
Lasso training MSE: 0.1415
R^2 score Lasso: 0.9568739343057595 with alpha: 0.01


In [143]:
def create_Bayesian_Ridge_pipeline():
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        linear_model.BayesianRidge()
    )

# Check it worked
alpha=0.1
pipe_Bayesian_ridge = create_Bayesian_Ridge_pipeline()
pipe_Bayesian_ridge.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe_Bayesian_ridge.predict(X_train))
print(f'Baysian Ridge training MSE: {mean_squared_error(y_train, pipe_Bayesian_ridge.predict(X_train)):.4f}')
MSE_scores["Bayesian Ridge with alpha 0.01"] = MSE

y_pred = pipe_Bayesian_ridge.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Baysian Ridge: {accuracy}")
R2_scores["Bayesian Ridge with alpha 0.01"] = accuracy

Baysian Ridge training MSE: 0.1125
R^2 score Baysian Ridge: 0.9232099789238918


In [148]:
#Constant that multiplies the penalty term. Defaults to 1.0. alpha = 0 is equivalent to an ordinary least square, 
# solved by LinearRegression. For numerical reasons, using alpha = 0 
# with the LassoLars object is not advised and you should prefer the LinearRegression object.

def create_LassoLARS_pipeline(alpha):
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        linear_model.LassoLars(alpha)
    )

# Check it worked
pipe_Lasso_Lars = create_LassoLARS_pipeline(alpha=1)
pipe_Lasso_Lars.fit(X_train, y_train)
MSE = mean_squared_error(y_train, pipe_Lasso_Lars.predict(X_train))
print(f'Lasso Lars training MSE: {mean_squared_error(y_train, pipe_Lasso_Lars.predict(X_train)):.4f}')
MSE_scores["Lasso LARS with alpha 1"] = MSE

y_pred = pipe_Lasso_Lars.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"R^2 score Lasso LARS: {accuracy}")
R2_scores["Lasso LARS with alpha 1"] = accuracy

Lasso Lars training MSE: 2.8195
R^2 score Lasso LARS: 0.6158306179169015


In [149]:
R2_scores

{'Ridge': 0.936354585225137,
 'KRR': 0.936316630905189,
 'Linear Regression': 0.8887282513832646,
 'Lasso with alpha 0.1': 0.938032388831701,
 'Lasso with alpha 0.01': 0.9568739343057595,
 'Bayesian Ridge with alpha 0.01': 0.9232099789238918,
 'Lasso LARS with alpha 1': 0.6158306179169015}

In [ ]:
sorted_r2 = {key: value for key, 
               value in reversed(sorted(R2_scores.items(), 
                               key=lambda item: item[1]))}

#We sort in reverse order because the best one is the one closest to 1
sorted_r2

{'Lasso with alpha 0.01': 0.9568739343057595,
 'Lasso with alpha 0.1': 0.938032388831701,
 'Ridge': 0.936354585225137,
 'KRR': 0.936316630905189,
 'Bayesian Ridge with alpha 0.01': 0.9232099789238918,
 'Linear Regression': 0.8887282513832646,
 'Lasso LARS with alpha 1': 0.6158306179169015}

In [150]:
MSE_scores

{'Ridge': 0.07351334908908323,
 'KRR': 0.06684742523993144,
 'Linear Regression': 0.0726097134942626,
 'Lasso with alpha 0.1': 0.45208484048813335,
 'Lasso with alpha 0.01': 0.1415092600571354,
 'Bayesian Ridge with alpha 0.01': 0.11245699029279299,
 'Lasso LARS with alpha 1': 2.8194502055262016}

In [ ]:
sorted_MSE = {key: value for key, 
               value in sorted(MSE_scores.items(), 
                               key=lambda item: item[1])}

# want the one closest to 0 MSE 
sorted_MSE

{'KRR': 0.06684742523993144,
 'Linear Regression': 0.0726097134942626,
 'Ridge': 0.07351334908908323,
 'Bayesian Ridge with alpha 0.01': 0.11245699029279299,
 'Lasso with alpha 0.01': 0.1415092600571354,
 'Lasso with alpha 0.1': 0.45208484048813335,
 'Lasso LARS with alpha 1': 2.8194502055262016}

For MHLTH the best answer is KRR or Ridge

In [ ]:
#Need to create a column for binary classification
# to do: figure out what we are predicting, scores MHLTH need to be discrete binary 

def create_logistic_regression_pipeline():
    return make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

# Checking it works
#LR_pipe = create_logistic_regression_pipeline()
#LR_pipe.fit(X_train, y_train)
#y_pred = LR_pipe.predict(X_test)
#accuracy = accuracy_score(y_test, y_pred)
#print(f"Accuracy Logistic Regression: {accuracy}")